In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from pathlib import Path
import json

class RunerKUParser:
    """
    Парсер для датской базы рунических надписей runer.ku.dk
    База содержит ~900 датских рунических надписей
    """
    
    def __init__(self):
        self.base_url = "https://runer.ku.dk"
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5,da;q=0.3",
        })
        self.images_dir = Path("runer_ku_images")
        self.images_dir.mkdir(exist_ok=True)
        self.stats = {
            'total_items': 0,
            'parsed': 0,
            'with_images': 0,
            'errors': 0
        }
    
    def normalize_url(self, url):
        """
        Нормализация URL для runer.ku.dk
        Сайт использует q.php?p=... для доступа к страницам
        """
        if not url:
            return url
        
        # Если уже правильный формат
        if 'q.php?p=' in url:
            return url
        
        # Если формат ?p=...
        if url.startswith('?p='):
            return f"{self.base_url}/q.php{url}"
        
        # Если относительный путь без query string
        if not url.startswith('http'):
            # Убираем начальный слэш если есть
            path = url.lstrip('/')
            return f"{self.base_url}/q.php?p={path}"
        
        # Если полный URL но без q.php
        if self.base_url in url and 'q.php' not in url:
            # Извлекаем путь после домена
            path = url.replace(self.base_url, '').lstrip('/?')
            if path.startswith('p='):
                return f"{self.base_url}/q.php?{path}"
            else:
                return f"{self.base_url}/q.php?p={path}"
        
        return url
    
    def get_all_items_list(self):
        """
        Получить список всех предметов со страницы Genstande (Objects)
        """
        url = f"{self.base_url}/q.php?p=runer/genstande"
        
        print(f"📥 Загружаем список предметов из: {url}")
        
        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            items = []
            
            # Ищем все ссылки на предметы
            # Формат: <a href="?p=runer/genstande/genstand/ID">Название</a>
            for link in soup.find_all('a', href=True):
                href = link['href']
                if 'genstand' in href and href.count('/') > 2:
                    # Извлекаем ID предмета
                    match = re.search(r'genstand/(\d+)', href)
                    if match:
                        item_id = match.group(1)
                        item_name = link.get_text(strip=True)
                        
                        # Нормализуем URL
                        full_url = self.normalize_url(href)
                        
                        items.append({
                            'id': item_id,
                            'name': item_name,
                            'url': full_url
                        })
            
            # Удаляем дубликаты по ID
            seen_ids = set()
            unique_items = []
            for item in items:
                if item['id'] not in seen_ids:
                    seen_ids.add(item['id'])
                    unique_items.append(item)
            
            print(f"✓ Найдено предметов: {len(unique_items)}")
            return unique_items
            
        except Exception as e:
            print(f"✗ Ошибка при загрузке списка: {e}")
            return []
    
    def parse_item_page(self, item_url, item_id, item_name):
        """
        Парсинг страницы отдельного предмета - данные в HTML таблицах
        """
        try:
            response = self.session.get(item_url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            data = {
                'id': item_id,
                'name': item_name,
                'url': item_url,
                'dk_number': None,
                'sb_number': None,
                'type': None,
                'material_main': None,
                'material_sub': None,
                'dating': None,
                'dating_certainty': None,
                'dating_comment': None,
                'find_circumstances': None,
                'find_place': None,
                'find_place_certainty': None,
                'find_year': None,
                'owner_institution': None,
                'storage_location': None,
                'condition': None,
                'transliteration': None,
                'transcription': None,
                'translation_danish': None,
                'translation_english': None,
                'translation_comment': None,
                'reading_comment': None,
                'rune_typology': None,
                'rune_typology_certainty': None,
                'rune_height': None,
                'language_type': None,
                'inscription_placement': None,
                'script_arrangement': None,
                'ornamented': None,
                'separator_type': None,
                'archaeological_period': None,
                'other_number': None,
                'images': [],
                'literature': []
            }
            
            # Извлекаем заголовок страницы (содержит DK номер и название)
            h1 = soup.find('h1')
            if h1:
                title_text = h1.get_text(strip=True)
                # Извлекаем DK-номер (формат: "NJy 19: Название")
                dk_match = re.search(r'^([A-Z][A-Za-z]+\s+\d+):', title_text)
                if dk_match:
                    data['dk_number'] = dk_match.group(1)
            
            # Словарь маппинга датских полей на английские
            field_mapping = {
                # Genstand (Object) секция
                'Titel': 'title',
                'Fundomstændigheder': 'find_circumstances',
                'Genstandstype': 'type',
                'Ejerinstitution': 'owner_institution',
                'Opbevaringssted': 'storage_location',
                'Sb-nr.': 'sb_number',
                'Datering': 'dating',
                'Dateringssikkerhed': 'dating_certainty',
                'Dateringskommentar': 'dating_comment',
                'Fundsted': 'find_place',
                'Fundstedssikkerhed': 'find_place_certainty',
                'Fundår': 'find_year',
                'Arkæologisk periode': 'archaeological_period',
                'Overordnet materiale': 'material_main',
                'Underordnet materiale': 'material_sub',
                'Tilstand': 'condition',
                
                # Indskrift (Inscription) секция
                'DK nr.': 'dk_number_alt',
                'Dansk oversættelse': 'translation_danish',
                'English translation': 'translation_english',
                'Translitteration': 'transliteration',
                'Transkription': 'transcription',
                'Ornamenteret': 'ornamented',
                'Oversættelses- og sagkommentar': 'translation_comment',
                'Læsningskommentar': 'reading_comment',
                'Runetypologi': 'rune_typology',
                'Runetypologisikkerhed': 'rune_typology_certainty',
                'Sprogtype(r)': 'language_type',
                'Runehøjde': 'rune_height',
                'Skilletegnstype(r)': 'separator_type',
                'Skriftordning': 'script_arrangement',
                'Indskriftplacering': 'inscription_placement',
                'Andet nr.': 'other_number'
            }
            
            # Парсим ТАБЛИЦЫ (основной метод для этого сайта)
            for table in soup.find_all('table', class_='gefin-datatable'):
                for row in table.find_all('tr'):
                    cells = row.find_all(['th', 'td'])
                    if len(cells) >= 2:
                        # Первая ячейка - название поля
                        field_name = cells[0].get_text(strip=True)
                        # Вторая ячейка - значение
                        field_value = cells[1].get_text(strip=True)
                        
                        # Ищем соответствие в маппинге
                        if field_name in field_mapping:
                            english_field = field_mapping[field_name]
                            
                            # Обрабатываем специальные случаи
                            if field_value:
                                if field_value in ['Nej', 'Ja']:
                                    field_value = 'No' if field_value == 'Nej' else 'Yes'
                                
                                # Для DK nr. берем только текст без ссылки
                                if field_name == 'DK nr.':
                                    dk_span = cells[1].find('span', class_='runer-dknr')
                                    if dk_span:
                                        field_value = dk_span.get_text(strip=True)
                                
                                # Для полей со ссылками извлекаем только текст
                                link = cells[1].find('a')
                                if link and field_name in ['Fundsted', 'Ejerinstitution']:
                                    field_value = link.get_text(strip=True)
                                
                                # Для English translation проверяем div с lang="en"
                                if field_name == 'English translation':
                                    en_div = cells[1].find('div', attrs={'lang': 'en'})
                                    if en_div:
                                        field_value = en_div.get_text(strip=True)
                                        if not field_value:
                                            field_value = None
                                
                                if field_value:
                                    data[english_field] = field_value
            
            # Если DK номер найден в таблице, используем его
            if data.get('dk_number_alt'):
                data['dk_number'] = data['dk_number_alt']
                del data['dk_number_alt']
            
            # Извлекаем изображения более умно
            for img in soup.find_all('img'):
                img_src = img.get('src', '')
                
                # Пропускаем служебные изображения
                skip_patterns = ['logo', 'icon', 'button', 'banner', 'menu', 'ku.dk/grafik/globalmenu']
                if any(skip in img_src.lower() for skip in skip_patterns):
                    continue
                
                if img_src:
                    # Формируем полный URL
                    if img_src.startswith('http'):
                        img_url = img_src
                    elif img_src.startswith('/'):
                        img_url = f"{self.base_url}{img_src}"
                    else:
                        img_url = f"{self.base_url}/{img_src}"
                    
                    # Получаем информацию об изображении
                    img_title = img.get('title', img.get('alt', ''))
                    
                    # Проверяем, что это не дубликат
                    if not any(i['url'] == img_url for i in data['images']):
                        data['images'].append({
                            'url': img_url,
                            'title': img_title
                        })
            
            # Также проверяем ссылки на изображения
            for link in soup.find_all('a', href=True):
                href = link['href']
                if any(ext in href.lower() for ext in ['.jpg', '.jpeg', '.png', '.gif', '.tif', '.tiff']):
                    if href.startswith('http'):
                        img_url = href
                    elif href.startswith('/'):
                        img_url = f"{self.base_url}{href}"
                    else:
                        img_url = f"{self.base_url}/{href}"
                    
                    # Проверяем, не добавлено ли уже это изображение
                    if not any(img['url'] == img_url for img in data['images']):
                        data['images'].append({
                            'url': img_url,
                            'title': link.get_text(strip=True)
                        })
            
            # Извлекаем литературные ссылки
            # Ищем секцию "Litteratur" или "Literature"
            for heading in soup.find_all(['h2', 'h3', 'h4', 'strong', 'b']):
                if heading and re.search(r'Litteratur|Literature|Referencer', heading.get_text(), re.IGNORECASE):
                    # Берем текст после этого заголовка
                    current = heading.next_sibling
                    lit_count = 0
                    
                    while current and lit_count < 20:
                        if hasattr(current, 'name'):
                            # Если встретили новый заголовок - останавливаемся
                            if current.name in ['h1', 'h2', 'h3', 'h4']:
                                break
                            text = current.get_text(strip=True)
                        else:
                            text = str(current).strip()
                        
                        # Добавляем только содержательные строки
                        if text and len(text) > 20 and not text.startswith('http'):
                            data['literature'].append(text)
                            lit_count += 1
                        
                        current = current.next_sibling
                    break
            
            return data
            
        except Exception as e:
            print(f"   ✗ Ошибка парсинга {item_url}: {e}")
            self.stats['errors'] += 1
            return None
    
    def download_image(self, img_data, item_id, img_index):
        """
        Скачивание изображения
        """
        try:
            img_url = img_data['url']
            response = self.session.get(img_url, timeout=30, stream=True)
            
            if response.status_code == 200:
                # Определяем расширение
                content_type = response.headers.get('Content-Type', '')
                ext_map = {
                    'image/jpeg': 'jpg',
                    'image/png': 'png',
                    'image/gif': 'gif',
                    'image/tiff': 'tif'
                }
                ext = ext_map.get(content_type, 'jpg')
                
                # Если не определилось, берем из URL
                if ext == 'jpg':
                    url_ext = img_url.split('.')[-1].split('?')[0].lower()
                    if url_ext in ['jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff']:
                        ext = url_ext
                
                # Формируем безопасное имя файла
                safe_id = re.sub(r'[^\w\-]', '_', str(item_id))
                filename = self.images_dir / f"{safe_id}_{img_index}.{ext}"
                
                # Сохраняем файл
                with open(filename, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                file_size = filename.stat().st_size
                return {
                    'local_path': str(filename),
                    'size_kb': file_size // 1024,
                    'title': img_data.get('title', '')
                }
        except Exception as e:
            print(f"   ⚠ Ошибка загрузки изображения: {e}")
        
        return None
    
    def scrape_all(self, limit=None, delay=1, download_images=True):
        """
        Полное скачивание базы данных
        
        Параметры:
        - limit: ограничение количества записей (None = все)
        - delay: задержка между запросами (секунды)
        - download_images: скачивать ли изображения
        """
        print("=" * 60)
        print("🇩🇰 Парсер датской базы рунических надписей")
        print("=" * 60)
        
        # Получаем список всех предметов
        items = self.get_all_items_list()
        
        if not items:
            print("❌ Не удалось получить список предметов")
            return []
        
        self.stats['total_items'] = len(items)
        
        if limit:
            items = items[:limit]
            print(f"⚡ Ограничение: будет обработано {limit} записей\n")
        
        results = []
        
        # Парсим каждый предмет
        for idx, item in enumerate(items, 1):
            print(f"\n[{idx}/{len(items)}] {item['name']}")
            print(f"   URL: {item['url']}")
            
            # Парсим страницу
            data = self.parse_item_page(item['url'], item['id'], item['name'])
            
            if data:
                # Скачиваем изображения
                if download_images and data['images']:
                    print(f"   📸 Найдено изображений: {len(data['images'])}")
                    local_images = []
                    
                    for img_idx, img_data in enumerate(data['images']):
                        local_img = self.download_image(img_data, item['id'], img_idx)
                        if local_img:
                            local_images.append(local_img)
                            print(f"      ✓ Сохранено: {local_img['local_path']} ({local_img['size_kb']} KB)")
                    
                    data['local_images'] = local_images
                    
                    if local_images:
                        self.stats['with_images'] += 1
                
                results.append(data)
                self.stats['parsed'] += 1
                print(f"   ✓ Успешно обработано")
            
            # Задержка между запросами
            if idx < len(items):
                time.sleep(delay)
            
            # Промежуточная статистика каждые 50 записей
            if idx % 50 == 0:
                self.print_stats()
        
        return results
    
    def print_stats(self):
        """Вывод статистики"""
        print("\n" + "=" * 60)
        print("📊 СТАТИСТИКА")
        print("=" * 60)
        print(f"Всего предметов:      {self.stats['total_items']}")
        print(f"Обработано:           {self.stats['parsed']}")
        print(f"С изображениями:      {self.stats['with_images']}")
        print(f"Ошибок:               {self.stats['errors']}")
        print("=" * 60 + "\n")
    
    def create_dataframe(self, results):
        """
        Создание DataFrame из результатов
        """
        if not results:
            print("⚠ Нет данных для создания DataFrame")
            return pd.DataFrame()
        
        # Подготавливаем данные
        for r in results:
            # Преобразуем списки в строки
            if 'images' in r:
                r['images_count'] = len(r['images'])
                r['images_urls'] = '; '.join([img['url'] for img in r['images']])
                del r['images']
            
            if 'local_images' in r:
                r['local_images_count'] = len(r['local_images'])
                r['local_images_paths'] = '; '.join([img['local_path'] for img in r['local_images']])
                del r['local_images']
            
            if 'literature' in r:
                r['literature_count'] = len(r['literature'])
                r['literature_refs'] = ' | '.join(r['literature'])
                del r['literature']
            
            if 'inscriptions' in r:
                del r['inscriptions']
        
        df = pd.DataFrame(results)
        
        # Сортируем колонки
        priority_cols = ['id', 'name', 'dk_number', 'sb_number', 'type', 
                        'material_main', 'material_sub', 'dating', 'dating_certainty',
                        'find_place', 'find_place_certainty', 'find_year',
                        'transliteration', 'transcription',
                        'translation_danish', 'translation_english',
                        'rune_typology', 'rune_typology_certainty',
                        'language_type', 'inscription_placement',
                        'images_count', 'url']
        
        other_cols = [col for col in df.columns if col not in priority_cols]
        ordered_cols = [col for col in priority_cols if col in df.columns] + other_cols
        
        return df[ordered_cols]
    
    def export_results(self, results, output_prefix="runer_ku"):
        """
        Экспорт результатов в разные форматы
        """
        if not results:
            print("⚠ Нет данных для экспорта")
            return
        
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        
        # CSV
        df = self.create_dataframe(results)
        csv_file = f"{output_prefix}_{timestamp}.csv"
        df.to_csv(csv_file, index=False, encoding='utf-8-sig')
        print(f"✓ CSV сохранен: {csv_file}")
        
        # JSON
        json_file = f"{output_prefix}_{timestamp}.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✓ JSON сохранен: {json_file}")
        
        # Excel (опционально)
        try:
            excel_file = f"{output_prefix}_{timestamp}.xlsx"
            df.to_excel(excel_file, index=False, engine='openpyxl')
            print(f"✓ Excel сохранен: {excel_file}")
        except:
            print("⚠ Excel не удалось создать (установите openpyxl)")
        
        print(f"✓ Изображения в папке: {self.images_dir}/")
        
        return csv_file, json_file


# === ИСПОЛЬЗОВАНИЕ ===

if __name__ == "__main__":
    parser = RunerKUParser()
    
    print("\n╔═══════════════════════════════════════════╗")
    print("║  Runer.ku.dk Parser                       ║")
    print("║  Датская база рунических надписей        ║")
    print("╚═══════════════════════════════════════════╝\n")
    
    print("Выберите режим:")
    print("1 - Быстрый тест (10 записей)")
    print("2 - Средняя выборка (50 записей)")
    print("3 - Полная база (~900 записей)")
    
    choice = input("\nВаш выбор (1/2/3): ").strip()
    
    if choice == '1':
        print("\n🧪 Режим: Быстрый тест\n")
        results = parser.scrape_all(limit=10, delay=1, download_images=True)
    
    elif choice == '2':
        print("\n📊 Режим: Средняя выборка\n")
        results = parser.scrape_all(limit=50, delay=1, download_images=True)
    
    elif choice == '3':
        print("\n🌍 Режим: Полная база\n")
        print("⚠️  Это займет ~30-60 минут")
        confirm = input("Продолжить? (yes/no): ")
        if confirm.lower() != 'yes':
            print("Отменено")
            exit()
        results = parser.scrape_all(limit=None, delay=1.5, download_images=True)
    
    else:
        print("❌ Неверный выбор")
        exit()
    
    # Сохраняем результаты
    if results:
        parser.print_stats()
        
        print("\n" + "=" * 60)
        print("💾 СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
        print("=" * 60)
        
        csv_file, json_file = parser.export_results(results)
        
        # Показываем примеры данных
        df = parser.create_dataframe(results)
        
        print("\n" + "=" * 60)
        print("📋 ПРИМЕРЫ ДАННЫХ")
        print("=" * 60)
        print(f"\nВсего записей: {len(df)}")
        print(f"Колонок: {len(df.columns)}")
        
        if not df.empty:
            print("\nПримеры записей:")
            display_cols = ['name', 'dk_number', 'dating', 'find_place', 'images_count']
            available_cols = [col for col in display_cols if col in df.columns]
            print(df[available_cols].head(10))
            
            print("\n📊 Статистика по типам:")
            if 'type' in df.columns:
                print(df['type'].value_counts().head(10))
            
            print("\n📊 Статистика по местам находок:")
            if 'find_place' in df.columns:
                print(df['find_place'].value_counts().head(10))
        
        print("\n✅ Готово!")
    else:
        print("\n❌ Не удалось получить данные")


╔═══════════════════════════════════════════╗
║  Runer.ku.dk Parser                       ║
║  Датская база рунических надписей        ║
╚═══════════════════════════════════════════╝

Выберите режим:
1 - Быстрый тест (10 записей)
2 - Средняя выборка (50 записей)
3 - Полная база (~900 записей)

🌍 Режим: Полная база

⚠️  Это займет ~30-60 минут
Отменено
🇩🇰 Парсер датской базы рунических надписей
📥 Загружаем список предметов из: https://runer.ku.dk/q.php?p=runer/genstande
✓ Найдено предметов: 1052

[1/1052] Absalons ringUk 3
   URL: https://runer.ku.dk/q.php?p=runer/genstande/genstand/407
   📸 Найдено изображений: 2
      ✓ Сохранено: runer_ku_images\407_0.jpg (31 KB)
      ✓ Сохранено: runer_ku_images\407_1.jpg (95 KB)
   ✓ Успешно обработано

[2/1052] Aggersborg-kalkristning 1NJy 19
   URL: https://runer.ku.dk/q.php?p=runer/genstande/genstand/735
   📸 Найдено изображений: 2
      ✓ Сохранено: runer_ku_images\735_0.jpg (31 KB)
      ✓ Сохранено: runer_ku_images\735_1.jpg (140 KB)
   ✓ У

: 

In [25]:
df.transliteration

0                                        þorKair -----
1                              uk ... s(a)ul ... a ...
2                                             t=y(i)-k
3                           a=u=æ m=a=r a=fs=h=t(u)=k-
4                                         a=u=e m=aria
5                                   ð(o)=rs=n : ku-(-)
6                                                  ...
7     (n)a=nnaba=r(i)s---mo=t---- : | (p)(a)(t)=(æ)(r)
8    g=u=þ : g=(ø)-æ : t=ho=rlic=h : o=r : hæræ : s...
9                                         ma=rgaræta :
Name: transliteration, dtype: object